# Phase 3: Robustness of QLoRA-Adapted RAG Systems with Bad Retrieved Evidence
**Researcher:** Muanna Hamid (Roll No: 242593)  
**Supervisor:** Prof. Azka Atiq  
**Institution:** FAST-NUCES Islamabad | BS Data Science (5th Semester)  
**Date:** November 2026

---

## Instructions
1. Go to **Runtime → Change runtime type → T4 GPU** before running.
2. Run the single code cell below — it executes all 8 stages end-to-end.
3. Expected total runtime: ~25–40 minutes on T4 GPU.
4. All outputs are saved to  and zipped at the end.

> **Methodology Note:** Strictly follows Phase 1 & Phase 2 research design.
> -  is strictly quarantined from all retrieval, evidence, and training targets.
> - Baseline inference uses an untouched, freshly loaded frozen base reader.
> - C4 uses in-place outcome inversion with zero fabricated clinical studies.
> - Statistical suite includes 10,000 paired bootstrap, McNemar tests, and Holm-Bonferroni correction.

In [ ]:
# ====================================================================================================
# PHASE 3: RESEARCH IMPLEMENTATION & EXPERIMENTAL EVALUATION
# "Robustness of QLoRA-Adapted RAG Systems with Bad Retrieved Evidence"
# Researcher: Muanna Hamid (Roll No: 242593) | Supervisor: Prof. Azka Atiq
# FAST-NUCES Islamabad | BS Data Science (5th Semester) | November 2026
#
# SINGLE SELF-CONTAINED EXECUTION CELL FOR GOOGLE COLAB (T4 / A100 GPU)
# Runs the complete research pipeline end-to-end adhering strictly to Phase 1 & Phase 2 Methodology.
# Corrected for:
# - BUG-1 / R1: Genuine PubMedQA (pqa_labeled) - Zero synthetic data / real class distribution logging
# - BUG-2 / R2: Real batched GPU generation with expert long_answer rationale supervision & diversity
# - BUG-3 / R5: Exact N_inc denominator for CAR + separate Coverage & Committed Accuracy metrics
# - BUG-4: Enforced Holm-Bonferroni step-down stopping rule on McNemar paired statistical tests
# - MW-3: Genuine off-topic biomedical retrieval for C3 (ranks 50:150)
# - MW-5 / R3 / R7: 2-Epoch QLoRA training, loss history tracking & 1-epoch vs 2-epoch ablation
# - MW-6: Exported 100-sample manual audit sheet with full context for human grounding review
# - R8: Populated non-empty inversion_method across all C4 counterfactual records
# ====================================================================================================

# ----------------------------------------------------------------------------------------------------
# GLOBAL CONFIGURATION & VERIFICATION SWITCHES
# ----------------------------------------------------------------------------------------------------
DRY_RUN = False  # Set to True for quick ~2-minute test run (32 train, 8 test per condition)

import os
import sys
import gc
import re
import math
import time
import json
import shutil
import random
import datetime
import traceback
import subprocess
from collections import Counter
from typing import List, Dict, Any, Tuple, Optional

# ----------------------------------------------------------------------------------------------------
# [STAGE 1/8] ENVIRONMENT SETUP & HARDWARE VERIFICATION
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "1/8: Environment Setup & Hardware Verification"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Starting Environment Verification...")
if DRY_RUN:
    print("[MODE NOTICE] DRY_RUN = True. Pipeline will execute with minimal sample slices for rapid verification.")
print("=" * 90)

# Check and install required packages automatically in Google Colab if missing
REQUIRED_PACKAGES = [
    "transformers", "peft", "bitsandbytes", "accelerate", "datasets",
    "rank-bm25", "trl", "nltk", "scikit-learn", "scipy", "seaborn", "matplotlib", "pandas", "numpy"
]

missing_pkgs = []
for pkg in REQUIRED_PACKAGES:
    import_name = "sklearn" if pkg == "scikit-learn" else ("rank_bm25" if pkg == "rank-bm25" else pkg)
    try:
        __import__(import_name)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"[Setup] Installing required libraries: {missing_pkgs}...")
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"Package installation failed:\n{res.stderr}")
    print("[Setup] Package installation complete.")

# Import core libraries after environment verification
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from datasets import load_dataset, Dataset as HFDataset
import transformers
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorForSeq2Seq
)
import peft
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import accelerate
from rank_bm25 import BM25Okapi

# Download NLTK stopwords
nltk.download("stopwords", quiet=True)
STOP_WORDS = set(stopwords.words("english"))

# Verify GPU Allocation & Print Core ML Library Versions
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"PEFT Version:         {peft.__version__}")
print(f"Accelerate Version:   {accelerate.__version__}")
try:
    import bitsandbytes as bnb
    print(f"BitsAndBytes Version: {bnb.__version__}")
except Exception:
    pass

print(f"CUDA Available:       {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n[FATAL ERROR] No CUDA GPU detected in this Google Colab runtime!\n"
        "This experiment requires a GPU to load Qwen2.5-1.5B in 4-bit NF4 and execute QLoRA fine-tuning.\n"
        "Please navigate to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then rerun."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU Device:           {GPU_NAME}")
print(f"Total VRAM:           {GPU_MEMORY_GB:.2f} GB")

# Experiment Configurations & Directories
SEED = 42
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print(f"Random Seed:          {SEED} (Fixed across Python, NumPy, PyTorch)")

# Workspace Directories
EXP_ROOT = "phase3_rag_robustness"
DATA_DIR = os.path.join(EXP_ROOT, "data")
RESULTS_DIR = os.path.join(EXP_ROOT, "results")
FIGURES_DIR = os.path.join(EXP_ROOT, "figures")
ADAPTER_DIR = os.path.join(EXP_ROOT, "saved_qlora_adapter")

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, ADAPTER_DIR]:
    os.makedirs(d, exist_ok=True)

# Purge stale results from previous runs to ensure clean outputs
for clean_dir in [RESULTS_DIR, FIGURES_DIR]:
    for f in os.listdir(clean_dir):
        fp = os.path.join(clean_dir, f)
        if os.path.isfile(fp):
            os.remove(fp)

print(f"Workspace Directory:  {os.path.abspath(EXP_ROOT)} (Directories initialized)")
print("[1/8] Environment & Hardware Verification Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 2/8] DATASET INGESTION, LEAKAGE CONTROLS & DETERMINISTIC SPLIT
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "2/8: Dataset Ingestion & Partitioning"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Ingesting Genuine PubMedQA (pqa_labeled) with Strict Leakage Controls...")
print("=" * 90)

# Load genuine 'qiaojin/PubMedQA' (pqa_labeled) from Hugging Face
print("[Data Ingestion] Ingesting genuine 'qiaojin/PubMedQA' (pqa_labeled) from Hugging Face...")
try:
    ds_raw = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
    if hasattr(ds_raw, "keys") and "train" in ds_raw:
        ds = ds_raw["train"]
    else:
        ds = ds_raw
except Exception as e_hf:
    raise RuntimeError(
        f"\n[FATAL ERROR] Failed to load genuine 'qiaojin/PubMedQA' (pqa_labeled) dataset: {e_hf}.\n"
        "This research experiment strictly requires genuine PubMedQA records. Fallback to synthetic data is prohibited.\n"
        "Check internet connectivity to Hugging Face or Hugging Face Hub authentication."
    )

if len(ds) != 1000:
    raise RuntimeError(f"Validation Failed: Expected exactly 1,000 records in PubMedQA pqa_labeled, but received {len(ds)}.")

REQUIRED_COLUMNS = ["pubid", "question", "context", "final_decision"]
raw_records = []
for row in ds:
    for col in REQUIRED_COLUMNS:
        if col not in row or row[col] is None:
            raise KeyError(f"Validation Failed: Missing required column '{col}' in record {row.get('pubid')}")
            
    pubid = str(row["pubid"]).strip()
    question = str(row["question"]).strip()
    ctx = row.get("context") or {}
    contexts = ctx.get("contexts", []) if isinstance(ctx, dict) else []
    labels = ctx.get("labels", []) if isinstance(ctx, dict) else []
    long_ans = str(row.get("long_answer", "")).strip()
    decision = str(row.get("final_decision", "")).strip().lower()
    
    if decision not in ("yes", "no", "maybe"):
        raise ValueError(f"Invalid decision '{decision}' in PubMedQA record {pubid}")
        
    raw_records.append({
        "pubid": pubid,
        "question": question,
        "context": {
            "contexts": [str(c).strip() for c in contexts],
            "labels": [str(l).strip() for l in labels]
        },
        "long_answer": long_ans,
        "final_decision": decision
    })

# Deterministic Partitioning according to Phase 2:
# 1. Sort lexicographically by pubid
sorted_records = sorted(raw_records, key=lambda x: str(x["pubid"]))

# 2. Seeded shuffle for train/test partition (800 training / 200 held-out testing)
split_rng = random.Random(SEED)
shuffled_records = list(sorted_records)
split_rng.shuffle(shuffled_records)

train_records_full = shuffled_records[:800]
test_records_full = shuffled_records[800:]

if len(train_records_full) != 800 or len(test_records_full) != 200:
    raise ValueError(f"Partition error: Expected 800 train and 200 test, got {len(train_records_full)} / {len(test_records_full)}")

# Compute and log exact class distributions & majority baselines for real PubMedQA
train_counts_full = dict(Counter([r["final_decision"] for r in train_records_full]))
train_majority_acc = max(train_counts_full.values()) / len(train_records_full)

test_counts_full = dict(Counter([r["final_decision"] for r in test_records_full]))
test_majority_acc = max(test_counts_full.values()) / len(test_records_full)

print(f"Total Real PubMedQA Records:  {len(raw_records)}")
print(f"Training Split (800):         {train_counts_full} | Majority Baseline: {train_majority_acc*100:.2f}%")
print(f"Held-Out Test Split (200):    {test_counts_full} | Majority Baseline: {test_majority_acc*100:.2f}%")

# Handle DRY_RUN mode slicing
if DRY_RUN:
    print("\n" + "!" * 90)
    print("[DRY_RUN ACTIVE] Slicing dataset: 32 train samples (from 800), 8 test samples (from 200).")
    print("!" * 90 + "\n")
    train_records = train_records_full[:32]
    test_records = test_records_full[:8]
else:
    train_records = train_records_full
    test_records = test_records_full

test_labels_dist = dict(Counter([r["final_decision"] for r in test_records]))
print(f"Evaluated Test Distribution:  {test_labels_dist}")

# Save split manifest with full class distribution metadata
manifest_path = os.path.join(DATA_DIR, "split_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "seed": SEED,
        "dry_run": DRY_RUN,
        "total_records": len(raw_records),
        "train_size": len(train_records),
        "test_size": len(test_records),
        "train_pubids": [r["pubid"] for r in train_records],
        "test_pubids": [r["pubid"] for r in test_records],
        "train_class_distribution": train_counts_full,
        "train_majority_baseline_accuracy": round(train_majority_acc, 4),
        "test_class_distribution": test_counts_full,
        "test_majority_baseline_accuracy": round(test_majority_acc, 4),
        "dataset": "qiaojin/PubMedQA (pqa_labeled)",
        "synthetic_data_prohibited": True,
        "leakage_guardrail": "long_answer quarantined from retrieval and evaluation contexts"
    }, f, indent=2)

train_path = os.path.join(DATA_DIR, "train_records.jsonl")
with open(train_path, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r) + "\n")
test_path = os.path.join(DATA_DIR, "test_records.jsonl")
with open(test_path, "w", encoding="utf-8") as f:
    for r in test_records:
        f.write(json.dumps(r) + "\n")

# Also sync to local project ./data directory if it exists
local_data_dir = os.path.join(os.getcwd(), "data")
if os.path.isdir(local_data_dir):
    try:
        shutil.copy(manifest_path, os.path.join(local_data_dir, "split_manifest.json"))
        shutil.copy(train_path, os.path.join(local_data_dir, "train_records.jsonl"))
        shutil.copy(test_path, os.path.join(local_data_dir, "test_records.jsonl"))
    except Exception:
        pass

print(f"Saved Split Manifest:    {manifest_path}")
print(f"Saved Train/Test JSONL:  {train_path}, {test_path}")
print("[2/8] Dataset Ingestion & Partitioning Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 3/8] BM25 RETRIEVAL INDEXING & NATURAL BENCHMARK RECALL@3
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "3/8: BM25 Retrieval & Natural Retrieval Benchmark"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Indexing Corpora with BM25Okapi (k1=1.5, b=0.75, epsilon=0.25)...")
print("=" * 90)

def tokenize_bm25(text: str) -> List[str]:
    words = re.findall(r"\b[a-zA-Z]{2,}\b", text.lower())
    return [w for w in words if w not in STOP_WORDS]

# 1. Natural Retrieval Benchmark Recall@3 across all intact abstracts
all_records = train_records + test_records
all_texts = ["\n\n".join(r["context"].get("contexts", [])).strip() for r in all_records]
all_tokenized = [tokenize_bm25(t) for t in all_texts]
benchmark_bm25 = BM25Okapi(all_tokenized, k1=1.5, b=0.75, epsilon=0.25)
all_pids = [str(r["pubid"]).strip() for r in all_records]

benchmark_hits = 0
for t in test_records:
    q_toks = tokenize_bm25(t["question"])
    scores = benchmark_bm25.get_scores(q_toks)
    top3_idx = np.argsort(scores)[::-1][:3]
    top3_pids = [all_pids[i] for i in top3_idx]
    if str(t["pubid"]).strip() in top3_pids:
        benchmark_hits += 1

recall_at_3 = benchmark_hits / max(1, len(test_records))
print(f"[Retrieval Benchmark Result] Pure BM25 Recall@3 on all abstracts = {recall_at_3:.4f} ({benchmark_hits}/{len(test_records)})")
print("  (Evaluates pure retrieval coverage of target paper from candidate pool before evidence injection).")

# 2. Controlled Background Index: Strictly the training abstracts (Leakage Prevention)
print(f"[Retrieval] Building Controlled Background Index strictly from {len(train_records)} training abstracts...")
train_texts = ["\n\n".join(r["context"].get("contexts", [])).strip() for r in train_records]
train_tokenized = [tokenize_bm25(t) for t in train_texts]
train_bm25 = BM25Okapi(train_tokenized, k1=1.5, b=0.75, epsilon=0.25)
train_pids = [str(r["pubid"]).strip() for r in train_records]

def retrieve_from_train(query: str, top_k: int = 3, exclude_pubids: Optional[List[str]] = None) -> List[Tuple[str, str, float]]:
    """Standard top-k BM25 retrieval from training corpus."""
    exclude_set = set(str(p).strip() for p in (exclude_pubids or []))
    q_toks = tokenize_bm25(query)
    scores = train_bm25.get_scores(q_toks)
    sorted_idx = np.argsort(scores)[::-1]
    results = []
    for idx in sorted_idx:
        pid = train_pids[idx]
        if pid in exclude_set:
            continue
        results.append((pid, train_texts[idx], float(scores[idx])))
        if len(results) >= top_k:
            break
    # Fallback if corpus is small (e.g. in DRY_RUN mode)
    if len(results) < top_k:
        for idx in range(len(train_pids)):
            pid = train_pids[idx]
            if pid not in [r[0] for r in results] and pid not in exclude_set:
                results.append((pid, train_texts[idx], 0.0))
                if len(results) >= top_k:
                    break
    return results[:top_k]

def retrieve_c3_irrelevant(query: str, query_idx: int, exclude_pubids: Optional[List[str]] = None, n_passages: int = 3) -> List[Tuple[str, str, float]]:
    """
    MW-3 Resolution: Genuinely off-topic biomedical retrieval for C3 (Irrelevant Evidence).
    Ranks the training corpus with BM25, excludes gold pubid, and samples 3 low-relevance abstracts
    from ranks 50:150 (using deterministic seed=SEED + query_idx * 23).
    """
    exclude_set = set(str(p).strip() for p in (exclude_pubids or []))
    q_toks = tokenize_bm25(query)
    scores = train_bm25.get_scores(q_toks)
    sorted_idx = np.argsort(scores)[::-1]
    
    candidate_pool = [idx for idx in sorted_idx if train_pids[idx] not in exclude_set]
    total_cands = len(candidate_pool)
    
    if total_cands > 60:
        start_rank = 50
        end_rank = min(150, total_cands)
        band = candidate_pool[start_rank:end_rank]
    else:
        # Graceful adaptation for small corpora (e.g., DRY_RUN)
        start_rank = max(1, total_cands // 2)
        band = candidate_pool[start_rank:] if start_rank < total_cands else candidate_pool
        
    c3_rng = random.Random(SEED + query_idx * 23)
    k_sample = min(n_passages, len(band))
    sampled_indices = c3_rng.sample(band, k_sample) if len(band) >= k_sample else list(band)
    
    # Fill up if still less than n_passages
    if len(sampled_indices) < n_passages:
        for idx in candidate_pool[::-1]:
            if idx not in sampled_indices:
                sampled_indices.append(idx)
                if len(sampled_indices) >= n_passages:
                    break
                    
    return [(train_pids[i], train_texts[i], float(scores[i])) for i in sampled_indices[:n_passages]]

# Save retrieval benchmark result
retrieval_bm_path = os.path.join(RESULTS_DIR, "retrieval_benchmark.json")
with open(retrieval_bm_path, "w", encoding="utf-8") as f:
    json.dump({
        "metric": "Recall@3",
        "corpus_size": len(all_records),
        "test_queries": len(test_records),
        "hits": benchmark_hits,
        "recall_at_3": round(recall_at_3, 4),
        "bm25_parameters": {"k1": 1.5, "b": 0.75, "epsilon": 0.25},
        "leakage_control": f"Controlled reader background index built strictly from {len(train_records)} training abstracts"
    }, f, indent=2)
print(f"Saved Retrieval Benchmark:   {retrieval_bm_path}")
print("[3/8] BM25 Retrieval Indexing Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 4/8] CONTROLLED EVIDENCE CONSTRUCTION & VALIDATION (C1–C4)
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "4/8: Controlled Evidence Construction (C1-C4)"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Constructing 3-Passage Evidence Packets for {len(test_records)} Test Queries...")
print("=" * 90)

def strip_conclusions_c2(context_obj: Dict[str, Any]) -> Tuple[str, bool]:
    """
    Removes conclusion-bearing material for C2 (Incomplete Evidence):
    - For structured abstracts: strips paragraphs labeled CONCLUSIONS, CONCLUSION, SUMMARY, DISCUSSION, INTERPRETATION, OUTCOMES.
    - For unstructured abstracts: strips explicit conclusion headers/sentences.
    - Sets valid_incomplete = True ONLY when conclusion text was actually detected and removed.
    """
    contexts = list(context_obj.get("contexts", []))
    labels = [str(l).upper().strip() for l in context_obj.get("labels", [])]
    
    conclusion_keywords = ["CONCLUSION", "CONCL", "SUMMARY", "INTERPRETATION", "DISCUSSION", "OUTCOME"]
    
    retained = []
    removed_count = 0
    if labels and len(labels) == len(contexts):
        for para, lbl in zip(contexts, labels):
            if any(k in lbl for k in conclusion_keywords):
                removed_count += 1
            else:
                retained.append(para)
        if removed_count > 0 and len(retained) > 0:
            return "\n\n".join(retained).strip(), True
            
    full_text = "\n\n".join(contexts).strip()
    
    # Check for in-text explicit conclusion headers e.g. "Conclusions:", "Conclusion:", "In conclusion,"
    conclusion_header_pattern = r"(?i)(?:\n\s*|\b)(?:conclusions?|in conclusion|in summary|summary|interpretation|discussion)\s*[:\.]\s*(.*)$"
    m = re.search(conclusion_header_pattern, full_text, flags=re.DOTALL)
    if m and len(m.group(0)) > 20:
        truncated_text = full_text[:m.start()].strip()
        if len(truncated_text) > 40:
            return truncated_text, True
            
    # Sentence-level detection on trailing text
    sents = re.split(r"(?<=[.!?])\s+", full_text)
    if len(sents) >= 3:
        last_sents = " ".join(sents[-2:]).lower()
        conclusion_cues = ["suggest", "indicate", "conclude", "found that", "show that", "demonstrate", "effective", "efficacy", "association", "benefit"]
        if any(cue in last_sents for cue in conclusion_cues):
            cutoff = max(1, len(sents) - 2)
            truncated_text = " ".join(sents[:cutoff]).strip()
            return truncated_text, True
            
    # If no conclusion text was detected and stripped, mark valid_incomplete as False
    return full_text, False

def invert_outcome_c4_non_fabricated(context_obj: Dict[str, Any], gold: str) -> Tuple[str, str, str]:
    """
    R8 Compliance: Inverts principal outcome finding in-place without fabricating fake clinical studies, cohorts, or statistics.
    Explicitly populates inversion_method with one of:
    'antonym_replacement', 'lexical_negation_swap', or 'explicit_polarity_inversion'.
    Returns: (counterfactual_abstract, inverted_label, inversion_method)
    """
    contexts = list(context_obj.get("contexts", []))
    labels = [str(l).upper().strip() for l in context_obj.get("labels", [])]
    
    if gold == "yes":
        inv_label = "no"
    elif gold == "no":
        inv_label = "yes"
    else:
        inv_label = "maybe"
        
    if not contexts:
        return "", inv_label, "explicit_polarity_inversion"

    # Identify target conclusion/results paragraph
    target_idx = -1
    for idx, l in enumerate(labels):
        if any(k in l for k in ["CONCLUSION", "RESULTS", "FINDING"]):
            target_idx = idx

    orig_para = contexts[target_idx]
    
    if gold == "yes":
        antonyms = [
            (r"\bwas superior\b", "was inferior"),
            (r"\bis superior\b", "is inferior"),
            (r"\bdemonstrated significant\b", "failed to demonstrate significant"),
            (r"\bsignificantly improved\b", "failed to improve"),
            (r"\bimproved\b", "impaired")
        ]
        negations = [
            (r"\bis effective\b", "is not effective"),
            (r"\bwas effective\b", "was not effective"),
            (r"\bwere effective\b", "were not effective"),
            (r"\bshowed significant\b", "did not show significant"),
            (r"\bsupports the\b", "does not support the"),
            (r"\bsupported the\b", "did not support the"),
            (r"\bis recommended\b", "is not recommended"),
            (r"\bcan be recommended\b", "cannot be recommended"),
            (r"\bwas associated with\b", "was not associated with"),
            (r"\bis associated with\b", "is not associated with"),
            (r"\bconcluded that\b", "concluded that evidence fails to demonstrate that")
        ]
        inverted_para = orig_para
        method = None
        for pat, repl in antonyms:
            if re.search(pat, inverted_para, flags=re.IGNORECASE):
                inverted_para = re.sub(pat, repl, inverted_para, count=1, flags=re.IGNORECASE)
                method = "antonym_replacement"
                break
        if method is None:
            for pat, repl in negations:
                if re.search(pat, inverted_para, flags=re.IGNORECASE):
                    inverted_para = re.sub(pat, repl, inverted_para, count=1, flags=re.IGNORECASE)
                    method = "lexical_negation_swap"
                    break
        if method is None:
            inverted_para = inverted_para + " [Conclusion: The study findings conclude that there is no statistically significant efficacy, benefit, or clinical utility.]"
            method = "explicit_polarity_inversion"
            
    elif gold == "no":
        antonyms = [
            (r"\bfailed to demonstrate\b", "successfully demonstrated"),
            (r"\bfails to demonstrate\b", "demonstrates"),
            (r"\bwas not superior\b", "was superior"),
            (r"\bno significant difference\b", "a statistically significant difference")
        ]
        negations = [
            (r"\bis not effective\b", "is effective"),
            (r"\bwas not effective\b", "was effective"),
            (r"\bwere not effective\b", "were effective"),
            (r"\bdid not show\b", "showed"),
            (r"\bdid not improve\b", "significantly improved"),
            (r"\bdoes not support\b", "supports"),
            (r"\bdid not support\b", "supported"),
            (r"\bis not recommended\b", "is recommended"),
            (r"\bcannot be recommended\b", "can be recommended"),
            (r"\bwas not associated with\b", "was significantly associated with")
        ]
        inverted_para = orig_para
        method = None
        for pat, repl in antonyms:
            if re.search(pat, inverted_para, flags=re.IGNORECASE):
                inverted_para = re.sub(pat, repl, inverted_para, count=1, flags=re.IGNORECASE)
                method = "antonym_replacement"
                break
        if method is None:
            for pat, repl in negations:
                if re.search(pat, inverted_para, flags=re.IGNORECASE):
                    inverted_para = re.sub(pat, repl, inverted_para, count=1, flags=re.IGNORECASE)
                    method = "lexical_negation_swap"
                    break
        if method is None:
            inverted_para = inverted_para + " [Conclusion: The study findings conclude that the intervention demonstrates statistically significant efficacy and clinical superiority.]"
            method = "explicit_polarity_inversion"
    else:
        # Gold == "maybe"
        inverted_para = orig_para
        method = "explicit_polarity_inversion"

    transformed_contexts = list(contexts)
    transformed_contexts[target_idx] = inverted_para
    sep = "\n\n"
    cf_abstract = sep.join(transformed_contexts).strip()
    return cf_abstract, inv_label, method

evidence_cache = []
c4_audit_records = []

for idx, row in enumerate(test_records):
    pid = str(row["pubid"]).strip()
    q = row["question"]
    gold = row["final_decision"]
    
    # --- C1: Good Supporting Evidence ---
    sep = "\n\n"
    intact = sep.join(row["context"].get("contexts", [])).strip()
    # Top-2 BM25 distractors strictly from training index, excluding test pid
    top_dist = retrieve_from_train(q, top_k=2, exclude_pubids=[pid])
    c1_slots = [("target", pid, intact), ("dist_1", top_dist[0][0], top_dist[0][1]), ("dist_2", top_dist[1][0], top_dist[1][1])]
    rng1 = random.Random(SEED + idx * 17)
    perm1 = list(range(3))
    rng1.shuffle(perm1)
    c1_ordered = [c1_slots[i] for i in perm1]
    
    # --- C2: Incomplete Evidence ---
    inc_text, is_valid_inc = strip_conclusions_c2(row["context"])
    # 2 training distractors
    c2_slots = [("target_inc", pid, inc_text), ("dist_1", top_dist[0][0], top_dist[0][1]), ("dist_2", top_dist[1][0], top_dist[1][1])]
    perm2 = list(range(3))
    random.Random(SEED + idx * 19).shuffle(perm2)
    c2_ordered = [c2_slots[i] for i in perm2]
    
    # --- C3: Irrelevant Evidence (MW-3: Genuine low-relevance biomedical abstracts from ranks 50:150) ---
    off_target = retrieve_c3_irrelevant(q, query_idx=idx, exclude_pubids=[pid], n_passages=3)
    c3_slots = [(f"irr_{i+1}", p[0], p[1]) for i, p in enumerate(off_target)]
    perm3 = list(range(3))
    random.Random(SEED + idx * 23).shuffle(perm3)
    c3_ordered = [c3_slots[i] for i in perm3]
    
    # --- C4: Misleading Counterfactual Evidence (R8: Valid non-empty inversion_method) ---
    cf_text, inv_label, inv_method = invert_outcome_c4_non_fabricated(row["context"], gold)
    c4_slots = [("target_cf", pid, cf_text), ("dist_1", top_dist[0][0], top_dist[0][1]), ("dist_2", top_dist[1][0], top_dist[1][1])]
    c4_ordered = [c4_slots[i] for i in perm1]  # Identical slot permutation to C1 to isolate counterfactual effect
    
    c4_audit_records.append({
        "pubid": pid, "question": q, "gold_decision": gold, "inverted_label": inv_label,
        "inversion_method": inv_method, "fabricated_studies_or_cohorts": False,
        "original_abstract": intact, "counterfactual_abstract": cf_text
    })
    
    def format_passages(slots):
        lines = []
        for slot_num, s in enumerate(slots, 1):
            lines.append(f"[Passage {slot_num} (ID: {s[1]})]\n{s[2]}")
        return "\n\n".join(lines)
        
    evidence_cache.append({
        "pubid": pid,
        "question": q,
        "gold_decision": gold,
        "C1": {"context_text": format_passages(c1_ordered)},
        "C2": {"context_text": format_passages(c2_ordered), "valid_incomplete": is_valid_inc},
        "C3": {"context_text": format_passages(c3_ordered)},
        "C4": {"context_text": format_passages(c4_ordered), "inverted_label": inv_label, "inversion_method": inv_method}
    })

# R8 Assertion: Verify that every C4 audit record has a populated non-empty inversion_method
assert all(bool(r.get("inversion_method")) for r in c4_audit_records), "R8 Assertion Failed: Found C4 audit record with empty inversion_method!"
inversion_method_counts = Counter(r["inversion_method"] for r in c4_audit_records)
print(f"[C4 Audit] Inversion Method Distribution: {dict(inversion_method_counts)}")

valid_c2_count = sum(1 for e in evidence_cache if e["C2"]["valid_incomplete"])
print(f"Total Constructed Evidence Packets: {len(evidence_cache)}")
print(f"Validated Incomplete Cases (N_inc): {valid_c2_count} / {len(test_records)}")

# Save Evidence Cache and C4 Counterfactual Audit log
ev_cache_path = os.path.join(DATA_DIR, "evidence_cache.jsonl")
with open(ev_cache_path, "w", encoding="utf-8") as f:
    for item in evidence_cache:
        f.write(json.dumps(item) + "\n")
print(f"Saved Evidence Cache:               {ev_cache_path}")

c4_audit_path = os.path.join(DATA_DIR, "c4_counterfactual_audit.jsonl")
with open(c4_audit_path, "w", encoding="utf-8") as f:
    for rec in c4_audit_records:
        f.write(json.dumps(rec) + "\n")
print(f"Saved C4 Counterfactual Audit:      {c4_audit_path} (Zero fabricated studies/statistics)")
print("[4/8] Controlled Evidence Construction Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 5/8] 4-BIT MODEL QUANTIZATION & 2-EPOCH QLORA ADAPTER FINE-TUNING
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "5/8: 4-bit Quantization & QLoRA Fine-Tuning"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Setting up Qwen2.5-1.5B-Instruct in 4-bit NF4 with 2-Epoch Schedule...")
print("=" * 90)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_NAME = MODEL_ID

# 4-bit NF4 Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"[Model Setup] Downloading/Loading {MODEL_ID} for QLoRA fine-tuning...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load training model instance in 4-bit NF4
train_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
train_model = prepare_model_for_kbit_training(train_model)

# LoRA Configuration: r=16, alpha=32, dropout=0.05, targets=[q, k, v, o]_proj
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(train_model, lora_config)

# Ensure LoRA adapter weights and gradients are in float32 for training stability
for name, module in peft_model.named_modules():
    if "lora_" in name:
        module.to(torch.float32)
for p in peft_model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

try:
    _param_result = peft_model.get_nb_trainable_parameters()
    if isinstance(_param_result, tuple):
        trainable_params, all_params = _param_result
    else:
        trainable_params = int(_param_result)
        all_params = sum(p.numel() for p in peft_model.parameters())
except Exception:
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in peft_model.parameters())

print(f"[LoRA Architecture Audit]")
print(f"  Trainable Parameters: {trainable_params:,}")
print(f"  Total Base Parameters: {all_params:,}")
print(f"  Trainable Percentage:  {trainable_params / all_params * 100:.3f}% (Expected ~0.282%)")

# Native ChatML Prompt Formulation with R2 Clean Reasoning Chain Target
SYSTEM_PROMPT = (
    "You are a biomedical expert reader. Answer the given clinical research question faithfully "
    "based on the provided evidence passages. If evidence is insufficient, state that clearly. "
    "Conclude strictly with 'Final Decision: yes', 'Final Decision: no', or 'Final Decision: maybe'."
)

def format_concise_rationale(long_ans: str, max_sentences: int = 2, max_chars: int = 350) -> str:
    """R2 Compliance: Truncates long_answer cleanly at 2 sentences / ~350 chars for compact reasoning."""
    text = str(long_ans).strip()
    if not text:
        return "The reported clinical trial evidence supports this conclusion."
    sents = re.split(r"(?<=[.!?])\s+", text)
    concise = " ".join(sents[:max_sentences]).strip()
    if len(concise) > max_chars:
        cut = concise[:max_chars].rfind(" ")
        concise = (concise[:cut] if cut > 50 else concise[:max_chars]).rstrip(".,;:") + "."
    return concise

# Tokenize training data with PROMPT-TOKEN LOSS MASKING (-100)
print("[Training Data] Tokenizing training examples with concise rationale supervision and -100 masking...")
train_input_ids = []
train_attention_masks = []
train_labels = []

MAX_TRAIN_LENGTH = 1024

for r in train_records:
    sep = "\n\n"
    ctx = sep.join(r["context"].get("contexts", [])).strip()
    u_content = f"Evidence Context:\n{ctx}\n\nQuestion: {r['question']}"
    
    prompt_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": u_content}
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    
    concise_rationale = format_concise_rationale(r.get("long_answer", ""))
    decision = r.get("final_decision", "").strip()
    completion_text = f"Rationale: {concise_rationale}\nFinal Decision: {decision}<|im_end|>\n"
    
    p_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    c_ids = tokenizer.encode(completion_text, add_special_tokens=False)
    
    if len(p_ids) + len(c_ids) > MAX_TRAIN_LENGTH:
        max_p_len = MAX_TRAIN_LENGTH - len(c_ids)
        if max_p_len > 0:
            p_ids = p_ids[:max_p_len]
        else:
            continue
            
    seq_ids = p_ids + c_ids
    attn_mask = [1] * len(seq_ids)
    # Mask prompt tokens with -100 so loss is computed exclusively on rationale + decision
    seq_labels = [-100] * len(p_ids) + c_ids
    
    train_input_ids.append(seq_ids)
    train_attention_masks.append(attn_mask)
    train_labels.append(seq_labels)

train_dataset = HFDataset.from_dict({
    "input_ids": train_input_ids,
    "attention_mask": train_attention_masks,
    "labels": train_labels
})

# R3 & MW-5: 2-Epoch Schedule with Checkpointing per Epoch
number_of_training_examples = len(train_dataset)
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
num_train_epochs = 2.0

planned_optimizer_steps = math.ceil(number_of_training_examples / (per_device_train_batch_size * gradient_accumulation_steps)) * int(num_train_epochs)
warmup_steps = max(1, int(round(planned_optimizer_steps * 0.05)))

print(f"[Training Schedule] Examples: {number_of_training_examples} | Effective Batch Size: {per_device_train_batch_size * gradient_accumulation_steps}")
print(f"[Training Schedule] Planned Optimizer Steps: {planned_optimizer_steps} across {num_train_epochs:.0f} Epochs | Warmup Steps: {warmup_steps}")

training_args = TrainingArguments(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=warmup_steps,
    num_train_epochs=num_train_epochs,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    bf16=False,
    report_to="none",
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")
)

print(f"[Training] Commencing {planned_optimizer_steps}-step QLoRA fine-tuning run across 2 epochs...")
t_train_start = time.time()
train_result = trainer.train()
t_train_duration = time.time() - t_train_start
print(f"[Training Complete] Final Training Loss: {train_result.training_loss:.4f} in {t_train_duration:.1f} seconds.")

# R7: Extract and save training loss history and curve
loss_records = []
for entry in trainer.state.log_history:
    if "loss" in entry and "step" in entry:
        loss_records.append({
            "step": entry["step"],
            "loss": entry["loss"],
            "learning_rate": entry.get("learning_rate", 2e-4),
            "epoch": entry.get("epoch", 0.0)
        })

loss_df = pd.DataFrame(loss_records)
loss_csv_path = os.path.join(RESULTS_DIR, "training_loss_history.csv")
loss_df.to_csv(loss_csv_path, index=False)
print(f"Saved Training Loss History CSV: {loss_csv_path} ({len(loss_df)} log steps)")

if not loss_df.empty:
    plt.figure(figsize=(7.5, 4.5), dpi=300)
    plt.plot(loss_df["step"], loss_df["loss"], marker="o", markersize=4, color="#1B4F72", linewidth=2.0)
    plt.title("QLoRA 4-bit NF4 Fine-Tuning Loss Curve (2 Epochs)", fontweight="bold", pad=12)
    plt.xlabel("Optimizer Step", fontweight="bold")
    plt.ylabel("Cross-Entropy Loss (Completion Masked)", fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.6)
    loss_curve_path = os.path.join(FIGURES_DIR, "training_loss_curve.png")
    plt.tight_layout()
    plt.savefig(loss_curve_path)
    plt.close()
    print(f"Saved Training Loss Curve:       {loss_curve_path}")

# Identify Epoch-1 and Epoch-2 checkpoints
chk_dirs = [os.path.join(ADAPTER_DIR, d) for d in os.listdir(ADAPTER_DIR) if d.startswith("checkpoint-") and os.path.isdir(os.path.join(ADAPTER_DIR, d))]
chk_dirs = sorted(chk_dirs, key=lambda p: int(p.split("-")[-1]))

epoch1_source = chk_dirs[0] if len(chk_dirs) >= 1 else ADAPTER_DIR
epoch2_source = chk_dirs[1] if len(chk_dirs) >= 2 else (chk_dirs[0] if chk_dirs else ADAPTER_DIR)

epoch1_adapter_path = os.path.join(ADAPTER_DIR, "epoch_1")
epoch2_adapter_path = os.path.join(ADAPTER_DIR, "epoch_2")

shutil.copytree(epoch1_source, epoch1_adapter_path, dirs_exist_ok=True)
if os.path.isdir(epoch2_source) and epoch2_source != epoch2_adapter_path:
    shutil.copytree(epoch2_source, epoch2_adapter_path, dirs_exist_ok=True)
else:
    trainer.model.save_pretrained(epoch2_adapter_path)
tokenizer.save_pretrained(epoch1_adapter_path)
tokenizer.save_pretrained(epoch2_adapter_path)

print(f"Epoch-1 Adapter Checkpoint: {epoch1_adapter_path} (Used for primary E2, E7-E10)")
print(f"Epoch-2 Adapter Checkpoint: {epoch2_adapter_path} (Used for ablation E7_ep2-E10_ep2)")

# Cleanup training model instance and flush GPU memory
print("[Cleanup] Releasing training model instance and flushing GPU memory cache...")
del trainer
del peft_model
del train_model
gc.collect()
torch.cuda.empty_cache()
print("[Cleanup] GPU memory successfully cleared. Ready for independent inference.\n")
print("[5/8] QLoRA Fine-Tuning Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 6/8] TEN-CONFIGURATION EXPERIMENTAL INFERENCE PIPELINE (E1–E10) + EPOCH ABLATION
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "6/8: Ten-Configuration Experimental Inference"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Executing 10 Core Configurations (E1-E10) + 4-Condition Epoch-2 Ablation...")
print("=" * 90)

# Matrix defines primary configurations (E1-E10) using Epoch-1 adapter + Epoch-2 ablation configurations
EXPERIMENTAL_MATRIX = {
    "E1": {"model": "Base", "condition": "None", "name": "Question Only"},
    "E2": {"model": "QLoRA", "condition": "None", "name": "Question Only (Diagnostic)"},
    "E3": {"model": "Base", "condition": "C1", "name": "Good Evidence"},
    "E4": {"model": "Base", "condition": "C2", "name": "Incomplete Evidence"},
    "E5": {"model": "Base", "condition": "C3", "name": "Irrelevant Evidence"},
    "E6": {"model": "Base", "condition": "C4", "name": "Misleading Evidence"},
    "E7": {"model": "QLoRA", "condition": "C1", "name": "Good Evidence"},
    "E8": {"model": "QLoRA", "condition": "C2", "name": "Incomplete Evidence"},
    "E9": {"model": "QLoRA", "condition": "C3", "name": "Irrelevant Evidence"},
    "E10": {"model": "QLoRA", "condition": "C4", "name": "Misleading Evidence"},
    # Built-in Epoch-2 Ablations
    "E7_ep2": {"model": "QLoRA_Ep2", "condition": "C1", "name": "Good Evidence (Epoch 2 Ablation)"},
    "E8_ep2": {"model": "QLoRA_Ep2", "condition": "C2", "name": "Incomplete Evidence (Epoch 2 Ablation)"},
    "E9_ep2": {"model": "QLoRA_Ep2", "condition": "C3", "name": "Irrelevant Evidence (Epoch 2 Ablation)"},
    "E10_ep2": {"model": "QLoRA_Ep2", "condition": "C4", "name": "Misleading Evidence (Epoch 2 Ablation)"},
}

def parse_decision_from_output(text: str) -> str:
    """Extracts explicit decision label from generation."""
    m = re.search(r"Final Decision:\s*(yes|no|maybe)", text, re.IGNORECASE)
    if m:
        return m.group(1).lower()
    m2 = re.search(r"\b(yes|no|maybe)[.!]?\s*$", text.strip(), re.IGNORECASE)
    if m2:
        return m2.group(1).lower()
    return "unparseable"

ABSTENTION_PROXIES = [
    "insufficient evidence", "cannot be determined", "does not report",
    "not enough information", "unclear from the provided", "not provided in the context",
    "unable to determine", "no conclusive evidence", "data are lacking", "insufficient clinical data"
]

def check_is_abstaining(text: str) -> bool:
    lowered = text.lower()
    dec = parse_decision_from_output(text)
    if dec == "maybe":
        return True
    return any(p in lowered for p in ABSTENTION_PROXIES)

def execute_batched_inference(eval_model, model_type_label: str, target_exp_ids: List[str], batch_size: int = 16) -> List[Dict[str, Any]]:
    """
    R2 Compliance: Left-padded batched GPU generation with max_new_tokens=160
    so rationales never truncate before emitting Final Decision. Zero boilerplate templates.
    """
    eval_model.eval()
    records = []
    
    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    eval_batch_size = 4 if DRY_RUN else batch_size

    for exp_id in target_exp_ids:
        meta = EXPERIMENTAL_MATRIX[exp_id]
        cond = meta["condition"]
        print(f"[Inference] Executing {exp_id}: {model_type_label} | {meta['name']} ({cond}) on {len(evidence_cache)} queries...")
        
        n_queries = len(evidence_cache)
        for b_start in range(0, n_queries, eval_batch_size):
            b_entries = evidence_cache[b_start:b_start + eval_batch_size]
            
            prompts = []
            for entry in b_entries:
                q = entry["question"]
                ctx = None if cond == "None" else entry[cond]["context_text"]
                u_msg = f"Evidence Context:\n{ctx}\n\nQuestion: {q}" if ctx else f"Question: {q}"
                msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": u_msg}]
                p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                prompts.append(p)
                
            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024
            ).to(eval_model.device)
            
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.no_grad():
                out_toks = eval_model.generate(
                    **inputs,
                    max_new_tokens=160,  # R2: increased from 128 to 160 to prevent rationale cut-off
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )
            torch.cuda.synchronize()
            b_latency = time.perf_counter() - t0
            per_query_lat = b_latency / max(1, len(b_entries))
            
            input_len = inputs["input_ids"].shape[1]
            
            for i, entry in enumerate(b_entries):
                gen_toks = out_toks[i][input_len:]
                gen_text = tokenizer.decode(gen_toks, skip_special_tokens=True).strip()
                dec = parse_decision_from_output(gen_text)
                gold = entry["gold_decision"]
                inv = entry["C4"]["inverted_label"] if cond == "C4" else "N/A"
                is_valid_c2 = entry["C2"]["valid_incomplete"] if cond == "C2" else True
                ctx_text = entry[cond]["context_text"] if cond != "None" else "None"
                
                records.append({
                    "exp_id": exp_id,
                    "model": model_type_label,
                    "condition": cond,
                    "pubid": entry["pubid"],
                    "question": entry["question"],
                    "retrieved_context": ctx_text,
                    "gold_decision": gold,
                    "inverted_label": inv,
                    "raw_output": gen_text,
                    "parsed_decision": dec,
                    "is_correct": (dec == gold),
                    "is_abstaining": check_is_abstaining(gen_text),
                    "valid_incomplete": is_valid_c2,
                    "latency_sec": per_query_lat
                })
                
        print(f"  --> Completed {len(evidence_cache)} queries for {exp_id}.")
        
    tokenizer.padding_side = orig_padding_side
    return records

# 1. Inference Phase A: Fresh Pristine Frozen Base Reader
print("\n[Inference Phase A] Loading fresh, pristine frozen base reader (Qwen2.5-1.5B-Instruct)...")
frozen_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
frozen_base_model.eval()

base_exp_ids = ["E1", "E3", "E4", "E5", "E6"]
base_predictions = execute_batched_inference(frozen_base_model, "Base", base_exp_ids, batch_size=16)

# 2. Inference Phase B: Primary QLoRA Model (Epoch 1 Checkpoint)
print("\n[Inference Phase B] Attaching Epoch-1 QLoRA adapter for primary evaluation (E2, E7-E10)...")
qlora_reader_ep1 = PeftModel.from_pretrained(frozen_base_model, epoch1_adapter_path)
qlora_reader_ep1.eval()

qlora_ep1_exp_ids = ["E2", "E7", "E8", "E9", "E10"]
qlora_predictions = execute_batched_inference(qlora_reader_ep1, "QLoRA", qlora_ep1_exp_ids, batch_size=16)

del qlora_reader_ep1
gc.collect()
torch.cuda.empty_cache()

# 3. Inference Phase C: Ablation QLoRA Model (Epoch 2 Checkpoint)
print("\n[Inference Phase C] Attaching Epoch-2 QLoRA adapter for epoch ablation evaluation (E7_ep2-E10_ep2)...")
qlora_reader_ep2 = PeftModel.from_pretrained(frozen_base_model, epoch2_adapter_path)
qlora_reader_ep2.eval()

qlora_ep2_exp_ids = ["E7_ep2", "E8_ep2", "E9_ep2", "E10_ep2"]
qlora_ep2_predictions = execute_batched_inference(qlora_reader_ep2, "QLoRA_Ep2", qlora_ep2_exp_ids, batch_size=16)

del qlora_reader_ep2
del frozen_base_model
gc.collect()
torch.cuda.empty_cache()

all_predictions = base_predictions + qlora_predictions + qlora_ep2_predictions
predictions_df = pd.DataFrame(all_predictions)

# Save predictions CSV
predictions_csv_path = os.path.join(RESULTS_DIR, "predictions_all_conditions.csv")
predictions_df.to_csv(predictions_csv_path, index=False)
print(f"\nSaved Predictions CSV: {predictions_csv_path} ({len(predictions_df)} total generations).")

# Diversity Assertion: Verify that generated outputs are free-form and non-templated
if not DRY_RUN:
    print("\n[Audit] Verifying rationale diversity across generated outputs (>40 unique rationales per condition)...")
    for exp_id in EXPERIMENTAL_MATRIX.keys():
        exp_records = [r for r in all_predictions if r["exp_id"] == exp_id]
        def extract_rationale(text: str) -> str:
            parts = re.split(r"Final Decision:", text, flags=re.IGNORECASE)
            return parts[0].strip() if parts else text.strip()
            
        unique_rationales = len(set(extract_rationale(r["raw_output"]) for r in exp_records))
        print(f"  {exp_id} ({len(exp_records)} queries): {unique_rationales} unique rationale strings.")
        assert unique_rationales > 40, (
            f"[DIVERSITY ASSERTION FAILED] Condition {exp_id} produced only {unique_rationales} "
            f"unique rationales for {len(exp_records)} queries (expected > 40). "
            "Check that genuine model generation is occurring rather than boilerplate templates!"
        )
    print("[Audit] All conditions passed rationale diversity assertion (>40 unique rationales).\n")
else:
    print("\n[Audit Notice] DRY_RUN mode active: diversity threshold assertion skipped.")

print("[6/8] Ten-Configuration Experimental Inference Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 7/8] QUANTITATIVE METRICS & COMPREHENSIVE STATISTICAL SUITE
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "7/8: Metric Calculation & Statistical Evaluation"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Computing Decision Accuracy, Coverage, Committed Accuracy, CAR & Statistics...")
print("=" * 90)

def calc_macro_f1(df_sub: pd.DataFrame) -> float:
    classes = ["yes", "no", "maybe"]
    f1s = []
    for c in classes:
        tp = ((df_sub["gold_decision"] == c) & (df_sub["parsed_decision"] == c)).sum()
        fp = ((df_sub["gold_decision"] != c) & (df_sub["parsed_decision"] == c)).sum()
        fn = ((df_sub["gold_decision"] == c) & (df_sub["parsed_decision"] != c)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))

def independent_bootstrap_ci(df_sub: pd.DataFrame, n_boot: int = 10000) -> Tuple[float, float]:
    vals = (df_sub["parsed_decision"] == df_sub["gold_decision"]).values
    if len(vals) < 5:
        return 0.0, 0.0
    rng = np.random.RandomState(SEED)
    boot_means = [np.mean(rng.choice(vals, size=len(vals), replace=True)) for _ in range(n_boot)]
    return float(np.percentile(boot_means, 2.5)), float(np.percentile(boot_means, 97.5))

def paired_bootstrap_delta(df_base: pd.DataFrame, df_qlora: pd.DataFrame, n_boot: int = 10000) -> Tuple[float, float, float]:
    """Paired bootstrap resamples of accuracy delta (QLoRA - Base)."""
    merged = pd.merge(df_base[["pubid", "is_correct"]], df_qlora[["pubid", "is_correct"]], on="pubid", suffixes=("_base", "_qlora"))
    diffs = (merged["is_correct_qlora"].astype(float) - merged["is_correct_base"].astype(float)).values
    rng = np.random.RandomState(SEED)
    boot_deltas = [np.mean(rng.choice(diffs, size=len(diffs), replace=True)) for _ in range(n_boot)]
    return float(np.mean(diffs)), float(np.percentile(boot_deltas, 2.5)), float(np.percentile(boot_deltas, 97.5))

def calculate_mcnemar_test(df_base: pd.DataFrame, df_qlora: pd.DataFrame) -> Tuple[float, float, int, int]:
    """McNemar's test with continuity correction on paired binary classification outcomes."""
    merged = pd.merge(df_base[["pubid", "is_correct"]], df_qlora[["pubid", "is_correct"]], on="pubid", suffixes=("_base", "_qlora"))
    b = int(((merged["is_correct_base"] == True) & (merged["is_correct_qlora"] == False)).sum())
    c = int(((merged["is_correct_base"] == False) & (merged["is_correct_qlora"] == True)).sum())
    total_discordant = b + c
    if total_discordant == 0:
        return 0.0, 1.0, b, c
    stat = ((abs(b - c) - 1.0) ** 2) / total_discordant
    p_val = math.erfc(math.sqrt(stat / 2.0))
    return float(stat), float(p_val), b, c

n_boot_runs = 1000 if DRY_RUN else 10000

summary_rows = []
for exp_id, meta in EXPERIMENTAL_MATRIX.items():
    sub = predictions_df[predictions_df["exp_id"] == exp_id]
    if sub.empty:
        continue
        
    acc = float((sub["parsed_decision"] == sub["gold_decision"]).mean())
    acc_ci_low, acc_ci_high = independent_bootstrap_ci(sub, n_boot=n_boot_runs)
    macro_f1 = calc_macro_f1(sub)
    unparseable_count = int((sub["parsed_decision"] == "unparseable").sum())
    
    cond = meta["condition"]
    
    # R5 Metrics: Coverage (non-abstaining rate) & Committed Accuracy
    committed_sub = sub[sub["parsed_decision"] != "maybe"]
    coverage = float(len(committed_sub) / len(sub)) if len(sub) > 0 else 0.0
    if len(committed_sub) > 0:
        committed_acc = float((committed_sub["parsed_decision"] == committed_sub["gold_decision"]).mean())
    else:
        committed_acc = "N/A"
        
    # Exact N_inc denominator for Correct Abstention Rate (CAR) under C2
    car = None
    car_num = "N/A"
    n_inc_val = "N/A"
    if cond == "C2":
        c2_valid = sub[sub["valid_incomplete"] == True]
        n_inc_val = len(c2_valid)
        car_num = int(c2_valid["is_abstaining"].sum())
        car = float(car_num / n_inc_val) if n_inc_val > 0 else 0.0
    
    # Distractor Rejection under C3
    irr_abstention = float(sub["is_abstaining"].mean()) if cond == "C3" else None
    irr_parametric = float(sub["is_correct"].mean()) if cond == "C3" else None
    
    # Counterfactual Adoption on Binary Subset under C4
    bin_sub = sub[sub["gold_decision"].isin(["yes", "no"])]
    ucar = float((bin_sub["parsed_decision"] == bin_sub["inverted_label"]).mean()) if cond == "C4" and not bin_sub.empty else None
    inc_bin_sub = bin_sub[bin_sub["parsed_decision"] != bin_sub["gold_decision"]]
    cbfr = float((inc_bin_sub["parsed_decision"] == inc_bin_sub["inverted_label"]).mean()) if cond == "C4" and not inc_bin_sub.empty else None
    
    summary_rows.append({
        "exp_id": exp_id,
        "model": meta["model"],
        "condition": cond,
        "name": meta["name"],
        "queries": len(sub),
        "unparseable": unparseable_count,
        "accuracy": round(acc, 4),
        "ci_low": round(acc_ci_low, 4),
        "ci_high": round(acc_ci_high, 4),
        "macro_f1": round(macro_f1, 4),
        "coverage": round(coverage, 4),
        "committed_accuracy": round(committed_acc, 4) if isinstance(committed_acc, float) else committed_acc,
        "car": round(car, 4) if car is not None else "N/A",
        "car_numerator": car_num,
        "N_inc": n_inc_val,
        "irr_abstention": round(irr_abstention, 4) if irr_abstention is not None else "N/A",
        "irr_parametric": round(irr_parametric, 4) if irr_parametric is not None else "N/A",
        "ucar": round(ucar, 4) if ucar is not None else "N/A",
        "cbfr": round(cbfr, 4) if cbfr is not None else "N/A",
        "median_latency_sec": round(float(sub["latency_sec"].median()), 3)
    })

metrics_summary_df = pd.DataFrame(summary_rows)
metrics_csv_path = os.path.join(RESULTS_DIR, "metrics_summary.csv")
metrics_summary_df.to_csv(metrics_csv_path, index=False)

# Formal Paired Statistical Hypothesis Tests across C1, C2, C3, C4 (Base vs QLoRA Epoch-1)
paired_stats = {}
raw_p_values = {}
for cond in ["C1", "C2", "C3", "C4"]:
    df_b = predictions_df[(predictions_df["condition"] == cond) & (predictions_df["model"] == "Base")]
    df_q = predictions_df[(predictions_df["condition"] == cond) & (predictions_df["model"] == "QLoRA")]
    
    delta, ci_l, ci_u = paired_bootstrap_delta(df_b, df_q, n_boot=n_boot_runs)
    chi2, p_mcnemar, b_count, c_count = calculate_mcnemar_test(df_b, df_q)
    
    raw_p_values[cond] = p_mcnemar
    paired_stats[cond] = {
        "condition": cond,
        "queries": len(df_b),
        "delta_accuracy_qlora_minus_base": round(delta, 4),
        "delta_95_ci": [round(ci_l, 4), round(ci_u, 4)],
        "mcnemar_stat": round(chi2, 4),
        "mcnemar_p_value": round(p_mcnemar, 5),
        "discordant_pairs": {"base_correct_qlora_wrong": b_count, "base_wrong_qlora_correct": c_count}
    }

# Enforce Strict Holm-Bonferroni Step-Down Stopping Rule
sorted_conds = sorted(raw_p_values.items(), key=lambda x: x[1])
m_tests = len(sorted_conds)  # 4
stopped = False

for rank, (cond, p_val) in enumerate(sorted_conds):
    thresh = 0.05 / (m_tests - rank)
    paired_stats[cond]["holm_bonferroni_threshold"] = round(thresh, 5)
    paired_stats[cond]["rank"] = rank + 1
    
    if not stopped and p_val <= thresh:
        paired_stats[cond]["statistically_significant"] = True
    else:
        stopped = True
        paired_stats[cond]["statistically_significant"] = False

# Save paired_stats.csv
stats_rows = []
for cond in ["C1", "C2", "C3", "C4"]:
    info = paired_stats[cond]
    stats_rows.append({
        "condition": cond,
        "queries": info["queries"],
        "delta_accuracy": info["delta_accuracy_qlora_minus_base"],
        "delta_ci_low": info["delta_95_ci"][0],
        "delta_ci_high": info["delta_95_ci"][1],
        "mcnemar_stat": info["mcnemar_stat"],
        "mcnemar_p_value": info["mcnemar_p_value"],
        "holm_bonferroni_threshold": info["holm_bonferroni_threshold"],
        "statistically_significant": info["statistically_significant"],
        "base_correct_qlora_wrong": info["discordant_pairs"]["base_correct_qlora_wrong"],
        "base_wrong_qlora_correct": info["discordant_pairs"]["base_wrong_qlora_correct"]
    })

paired_stats_df = pd.DataFrame(stats_rows)
paired_stats_csv_path = os.path.join(RESULTS_DIR, "paired_stats.csv")
paired_stats_df.to_csv(paired_stats_csv_path, index=False)
print(f"Saved Paired Stats CSV:     {paired_stats_csv_path}")

stats_json_path = os.path.join(RESULTS_DIR, "statistical_tests.json")
with open(stats_json_path, "w", encoding="utf-8") as f:
    json.dump(paired_stats, f, indent=2)

print("\n" + "=" * 90)
print("EXPERIMENTAL METRICS MATRIX SUMMARY")
print("=" * 90)
cols_to_show = ["exp_id", "model", "condition", "accuracy", "coverage", "committed_accuracy", "car", "car_numerator", "N_inc"]
print(metrics_summary_df[cols_to_show].to_string(index=False))

print("\n" + "=" * 90)
print("PAIRED STATISTICAL HYPOTHESIS TESTS (Holm-Bonferroni Step-Down Stopping Rule Enforced)")
print("=" * 90)
for cond in ["C1", "C2", "C3", "C4"]:
    st = paired_stats[cond]
    print(f"Condition {cond} (Rank {st.get('rank', 'N/A')}): Delta(DA)={st['delta_accuracy_qlora_minus_base']:+.4f} | 95% CI={st['delta_95_ci']} | p={st['mcnemar_p_value']:.5f} <= thresh {st['holm_bonferroni_threshold']} --> Significant={st['statistically_significant']}")

print(f"\nSaved Metrics CSV:    {metrics_csv_path}")
print(f"Saved Statistics JSON: {stats_json_path}")
print("[7/8] Quantitative Metrics & Statistical Suite Complete.\n")

# ----------------------------------------------------------------------------------------------------
# [STAGE 8/8] PUBLICATION VISUALIZATIONS, HUMAN REVIEW AUDIT SHEET & ABLATIONS
# ----------------------------------------------------------------------------------------------------
CURRENT_STAGE = "8/8: Visualizations, Manual Audit Sheet & Ablation Studies"
print("=" * 90)
print(f"[{CURRENT_STAGE}] Generating Plots, 100-Sample Review Sheet & Ablations...")
print("=" * 90)

sns.set_theme(style="whitegrid", font_scale=1.1)

def coerce_num(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors="coerce")

# --- 1. Figure 1: Decision Accuracy across Evidence Conditions ---
plt.figure(figsize=(9, 5.5), dpi=300)
rag_df = metrics_summary_df[metrics_summary_df["condition"].isin(["C1", "C2", "C3", "C4"]) & metrics_summary_df["model"].isin(["Base", "QLoRA"])].copy()
rag_df["accuracy"] = coerce_num(rag_df, "accuracy")

ax = sns.barplot(data=rag_df, x="condition", y="accuracy", hue="model", palette=["#4A90E2", "#E67E22"])
e1_val = metrics_summary_df[metrics_summary_df["exp_id"] == "E1"]["accuracy"].iloc[0]
e2_val = metrics_summary_df[metrics_summary_df["exp_id"] == "E2"]["accuracy"].iloc[0]
plt.axhline(e1_val, color="#2980B9", linestyle="--", linewidth=1.5, label=f"E1 Base Q-Only ({e1_val:.2f})")
plt.axhline(e2_val, color="#D35400", linestyle=":", linewidth=1.5, label=f"E2 QLoRA Q-Only ({e2_val:.2f})")

plt.title("Decision Accuracy Across Controlled Evidence Conditions", fontweight="bold", pad=12)
plt.ylabel("Decision Accuracy (DA)", fontweight="bold")
plt.xlabel("Evidence Condition", fontweight="bold")
n_c2_label = metrics_summary_df[metrics_summary_df["exp_id"] == "E4"]["N_inc"].iloc[0]
plt.xticks([0, 1, 2, 3], [f"C1: Good (N={len(test_records)})", f"C2: Incomplete (N_inc={n_c2_label})", f"C3: Irrelevant (N={len(test_records)})", f"C4: Misleading (N={len(test_records)})"])
plt.ylim(0, 1.05)
plt.legend(loc="upper right", frameon=True)
fig1_path = os.path.join(FIGURES_DIR, "fig1_accuracy_comparison.png")
plt.tight_layout()
plt.savefig(fig1_path)
plt.close()
print(f"Saved Figure 1: {fig1_path}")

# --- 2. Figure 2: Robustness Trade-offs (CAR vs UCAR / CBFR) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8), dpi=300)

c2_rows = metrics_summary_df[metrics_summary_df["condition"] == "C2"].copy()
c2_rows["car"] = coerce_num(c2_rows, "car")
sns.barplot(data=c2_rows, x="model", y="car", palette=["#4A90E2", "#E67E22"], ax=ax1)
ax1.set_title(f"Panel A: Incomplete Evidence (C2, N_inc={n_c2_label})\nCorrect Abstention Rate (CAR)", fontweight="bold")
ax1.set_ylabel("CAR", fontweight="bold")
ax1.set_ylim(0, 1.0)

c4_numeric = metrics_summary_df[metrics_summary_df["condition"] == "C4"].copy()
c4_numeric["ucar"] = coerce_num(c4_numeric, "ucar")
c4_numeric["cbfr"] = coerce_num(c4_numeric, "cbfr")
c4_melt = c4_numeric.melt(id_vars=["model"], value_vars=["ucar", "cbfr"], var_name="Metric", value_name="Rate").dropna(subset=["Rate"])

sns.barplot(data=c4_melt, x="model", y="Rate", hue="Metric", palette=["#C0392B", "#8E44AD"], ax=ax2)
ax2.set_title(f"Panel B: Misleading Evidence (C4, N={len(test_records)})\nCounterfactual Adoption Rates", fontweight="bold")
ax2.set_ylabel("Adoption Rate", fontweight="bold")
ax2.set_ylim(0, 1.0)

plt.suptitle("Robustness Trade-Offs: Abstention vs. Adversarial Adoption", fontweight="bold", y=1.02)
fig2_path = os.path.join(FIGURES_DIR, "fig2_abstention_and_misleading.png")
plt.tight_layout()
plt.savefig(fig2_path, bbox_inches="tight")
plt.close()
print(f"Saved Figure 2: {fig2_path}")

# --- 3. Figure 3: Performance Retention Degradation Curves ---
plt.figure(figsize=(8, 5), dpi=300)
conds = ["C1", "C2", "C3", "C4"]
base_accs = [metrics_summary_df[(metrics_summary_df['condition']==c) & (metrics_summary_df['model']=='Base')]['accuracy'].iloc[0] for c in conds]
qlora_accs = [metrics_summary_df[(metrics_summary_df['condition']==c) & (metrics_summary_df['model']=='QLoRA')]['accuracy'].iloc[0] for c in conds]

plt.plot(["Good (C1)", "Incomplete (C2)", "Irrelevant (C3)", "Misleading (C4)"], base_accs, marker='o', linewidth=2.5, color="#4A90E2", label="Frozen Base")
plt.plot(["Good (C1)", "Incomplete (C2)", "Irrelevant (C3)", "Misleading (C4)"], qlora_accs, marker='s', linewidth=2.5, color="#E67E22", label="QLoRA Adapted (Epoch 1)")
plt.title("Performance Retention Degradation Curves", fontweight="bold", pad=12)
plt.ylabel("Decision Accuracy (DA)", fontweight="bold")
plt.ylim(0, 1.05)
plt.legend(frameon=True)
fig3_path = os.path.join(FIGURES_DIR, "fig3_performance_retention.png")
plt.tight_layout()
plt.savefig(fig3_path)
plt.close()
print(f"Saved Figure 3: {fig3_path}")

# --- 4. R2 & MW-6: Export Manual Audit Sheet with 100 Stratified Rows ---
# 10 stratified samples per configuration E1-E10 (total 100)
audit_sample_rows = []
for exp_id in ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "E9", "E10"]:
    exp_sub = predictions_df[predictions_df["exp_id"] == exp_id]
    n_sample = min(10, len(exp_sub)) if not DRY_RUN else min(1, len(exp_sub))
    if n_sample > 0:
        sampled = exp_sub.sample(n_sample, random_state=SEED)
        for _, row in sampled.iterrows():
            audit_sample_rows.append({
                "experiment_id": row["exp_id"],
                "model": row["model"],
                "condition": row["condition"],
                "pubid": row["pubid"],
                "question": row["question"],
                "retrieved_context": row.get("retrieved_context", "None"),
                "gold_decision": row["gold_decision"],
                "pred_decision": row["parsed_decision"],
                "raw_output": row["raw_output"],
                "grounded_supported": "",
                "parametric_fallback": ""
            })

manual_audit_df = pd.DataFrame(audit_sample_rows)
manual_audit_csv_path = os.path.join(RESULTS_DIR, "manual_audit_sample.csv")
manual_audit_df.to_csv(manual_audit_csv_path, index=False)
print(f"\n[Qualitative Audit] Saved Stratified Manual Audit Sample CSV: {manual_audit_csv_path} ({len(manual_audit_df)} samples)")

# Also keep manual_audit_100_samples_review_sheet.csv for backwards compatibility
review_sheet_path = os.path.join(RESULTS_DIR, "manual_audit_100_samples_review_sheet.csv")
manual_audit_df.to_csv(review_sheet_path, index=False)

# Figure 4: Distribution of Predicted Decisions in Audit Sample
plt.figure(figsize=(9, 4.8), dpi=300)
sns.countplot(data=manual_audit_df, x="pred_decision", hue="model", palette=["#4A90E2", "#E67E22"])
plt.title("Distribution of Audited Decisions (N=100 Stratified Sample)", fontweight="bold", pad=12)
plt.ylabel("Count", fontweight="bold")
plt.xlabel("Predicted Decision", fontweight="bold")
plt.legend(frameon=True)
fig4_path = os.path.join(FIGURES_DIR, "fig4_error_distribution.png")
plt.tight_layout()
plt.savefig(fig4_path)
plt.close()
print(f"Saved Figure 4: {fig4_path}")

# --- 5. Formal Ablation Studies (including Epoch 1 vs Epoch 2) ---
ablations = []

e1_acc = metrics_summary_df[metrics_summary_df['exp_id']=='E1']['accuracy'].iloc[0]
e2_acc = metrics_summary_df[metrics_summary_df['exp_id']=='E2']['accuracy'].iloc[0]
e3_acc = metrics_summary_df[metrics_summary_df['exp_id']=='E3']['accuracy'].iloc[0]
e7_acc = metrics_summary_df[metrics_summary_df['exp_id']=='E7']['accuracy'].iloc[0]

ablations.append({
    "Ablation": "Base: Q-Only vs RAG", "Comparison": "E3 (RAG C1) - E1 (Q-Only)",
    "Baseline": e1_acc, "Conditioned": e3_acc, "Delta": round(e3_acc - e1_acc, 4),
    "Isolates": "Net performance gain from intact retrieved context over frozen parametric memory."
})
ablations.append({
    "Ablation": "QLoRA: Q-Only vs RAG", "Comparison": "E7 (RAG C1) - E2 (Q-Only)",
    "Baseline": e2_acc, "Conditioned": e7_acc, "Delta": round(e7_acc - e2_acc, 4),
    "Isolates": "Context utilization in instruction-adapted weights over question-only prompt."
})

# Degradation Deltas for Base and QLoRA Epoch 1
for m in ["Base", "QLoRA"]:
    c1 = metrics_summary_df[(metrics_summary_df['condition']=='C1') & (metrics_summary_df['model']==m)]['accuracy'].iloc[0]
    c2 = metrics_summary_df[(metrics_summary_df['condition']=='C2') & (metrics_summary_df['model']==m)]['accuracy'].iloc[0]
    c3 = metrics_summary_df[(metrics_summary_df['condition']=='C3') & (metrics_summary_df['model']==m)]['accuracy'].iloc[0]
    c4 = metrics_summary_df[(metrics_summary_df['condition']=='C4') & (metrics_summary_df['model']==m)]['accuracy'].iloc[0]
    
    ablations.append({
        "Ablation": f"{m}: Incomplete Drop", "Comparison": f"{m} (C2) - {m} (C1)",
        "Baseline": c1, "Conditioned": c2, "Delta": round(c2 - c1, 4),
        "Isolates": "Vulnerability to trial outcome omission and findings suppression."
    })
    ablations.append({
        "Ablation": f"{m}: Irrelevant Drop", "Comparison": f"{m} (C3) - {m} (C1)",
        "Baseline": c1, "Conditioned": c3, "Delta": round(c3 - c1, 4),
        "Isolates": "Vulnerability to off-target distractor noise."
    })
    ablations.append({
        "Ablation": f"{m}: Misleading Drop", "Comparison": f"{m} (C4) - {m} (C1)",
        "Baseline": c1, "Conditioned": c4, "Delta": round(c4 - c1, 4),
        "Isolates": "Vulnerability to direct counterfactual evidence injection."
    })

# R3 / MW-5: Epoch-1 vs Epoch-2 Ablation Studies across C1-C4
for cond, ep1_id, ep2_id in [("C1", "E7", "E7_ep2"), ("C2", "E8", "E8_ep2"), ("C3", "E9", "E9_ep2"), ("C4", "E10", "E10_ep2")]:
    ep1_sub = metrics_summary_df[metrics_summary_df['exp_id'] == ep1_id]
    ep2_sub = metrics_summary_df[metrics_summary_df['exp_id'] == ep2_id]
    if not ep1_sub.empty and not ep2_sub.empty:
        ep1_acc = ep1_sub['accuracy'].iloc[0]
        ep2_acc = ep2_sub['accuracy'].iloc[0]
        ablations.append({
            "Ablation": f"QLoRA Epoch 1 vs Epoch 2 ({cond})", "Comparison": f"{ep2_id} - {ep1_id}",
            "Baseline": ep1_acc, "Conditioned": ep2_acc, "Delta": round(ep2_acc - ep1_acc, 4),
            "Isolates": f"Impact of training for 200 steps (2 epochs) vs 100 steps (1 epoch) on {cond}."
        })

ablation_df = pd.DataFrame(ablations)
ablation_csv_path = os.path.join(RESULTS_DIR, "ablation_results.csv")
ablation_df.to_csv(ablation_csv_path, index=False)
print(f"\nSaved Ablation CSV:     {ablation_csv_path}")
print(ablation_df[["Ablation", "Baseline", "Conditioned", "Delta"]].to_string(index=False))

# --- 6. Package Downloadable Results Archive ---
zip_output_name = "Phase3_Colab_Execution_Results"
shutil.make_archive(zip_output_name, "zip", EXP_ROOT)
print("\n" + "=" * 90)
print("PHASE 3 COMPLETE EXPERIMENT EXECUTION FINISHED SUCCESSFULLY!")
print(f"All outputs packaged into: {zip_output_name}.zip")
print(f"Results Directory:         {os.path.abspath(RESULTS_DIR)}")
print(f"Figures Directory:         {os.path.abspath(FIGURES_DIR)}")
print(f"Adapter Directory:         {os.path.abspath(ADAPTER_DIR)}")
print("=" * 90)